In [ ]:
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [ ]:

loader = UnstructuredPDFLoader(
    file_path="./p.pdf",
    strategy="hi_res",   # Required for accurate coordinate detection
    mode="elements"      # Breaks PDF down into structural chunks instead of full pages
)

chunks = loader.load()



In [ ]:
import json
import numpy as np
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

def sanitize_coordinates(coords):
    """Recursively converts np.float64 coordinates to standard Python floats."""
    if isinstance(coords, dict):
        return {k: sanitize_coordinates(v) for k, v in coords.items()}
    elif isinstance(coords, (list, tuple)):
        return [sanitize_coordinates(v) for v in coords]
    elif isinstance(coords, (np.float64, np.float32)):
        return float(coords)
    return coords

cleaned_documents = []

for chunk in chunks:
    if not chunk.page_content.strip():
        continue
    
    text_to_embed = chunk.page_content
    metadata = chunk.metadata.copy() 
    
    if "coordinates" in metadata:
        # 1. Convert numpy types to basic Python types first
        clean_coords = sanitize_coordinates(metadata["coordinates"])
        
        # 2. CRITICAL FIX: Convert the dictionary to a JSON string so Chroma accepts it
        metadata["coordinates"] = json.dumps(clean_coords)
        
    cleaned_documents.append(
        Document(page_content=text_to_embed, metadata=metadata)
    )

In [ ]:
print(cleaned_documents[0])

In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vector_db = Chroma.from_documents(
    documents=cleaned_documents,
    embedding=embedding_model,
    persist_directory="./chroma_db",
    collection_metadata={"hnsw:space": "cosine"} 
)

In [ ]:
import json
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = Chroma(
    persist_directory="./chroma_db", 
    embedding_function=embedding_model
)


In [ ]:
import numpy as np

def retrieve_dynamic_section(query: str, k: int = 5):
    # 1. Get primary matches with relevance scores
    # similarity_search_with_relevance_scores returns a list of tuples: (Document, score)
    primary_matches = vector_db.similarity_search_with_relevance_scores(query, k=k)
    if not primary_matches:
        return []
    
    # 2. DYNAMICALLY CHOOSE THE BEST TARGET
    # Default to the absolute top match
    best_match = primary_matches[0][0]
    
    # Scan matches to find if there is a 'Title' that strongly belongs to the query context
    # This completely replaces the hardcoded keyword list
    for doc, score in primary_matches:
        category = doc.metadata.get("category")
        
        # If a later match is explicitly a layout 'Title' and still holds a strong 
        # semantic relationship to the query, pivot to it as our section root!
        if category == "Title" and score > 0.3:  # 0.3 is a standard baseline for cosine relevance
            best_match = doc
            break

    best_metadata = best_match.metadata
    
    # 3. Proceed with your sequential page scanning logic
    if best_metadata.get("category") == "Title":
        print(f"🎯 Dynamically localized section root: '{best_match.page_content}'")
        
        all_page_elements = vector_db.get(
            where={
                "$and": [
                    {"source": best_metadata.get("source")},
                    {"page_number": best_metadata.get("page_number")}
                ]
            }
        )
        
        documents_on_page = []
        for i in range(len(all_page_elements["documents"])):
            documents_on_page.append({
                "text": all_page_elements["documents"][i],
                "metadata": all_page_elements["metadatas"][i],
                "id": all_page_elements["ids"][i]
            })
            
        start_idx = None
        for idx, doc in enumerate(documents_on_page):
            if doc["metadata"].get("element_id") == best_metadata.get("element_id"):
                start_idx = idx
                break
                
        if start_idx is not None:
            context_chunks = [documents_on_page[start_idx]]
            target_parent_id = best_metadata.get("element_id")
            
            for next_doc in documents_on_page[start_idx + 1:]:
                next_metadata = next_doc["metadata"]
                
                if next_metadata.get("parent_id") == target_parent_id:
                    context_chunks.append(next_doc)
                    continue
                
                if next_metadata.get("category") == "Title":
                    break
                    
                context_chunks.append(next_doc)
                
            return context_chunks
            
    return [{
        "text": best_match.page_content,
        "metadata": best_metadata
    }]

In [ ]:
# queries = [
#     "What company launched the cloud modernization initiative?",
#     "When was the modernization project started?",
#     "How many physical servers existed before migration?",
#     "What was the monthly infrastructure cost before migration?",
#     "What is the current monthly infrastructure cost?",
#     "Who was the Lead Architect?",
#     "Who was the DevOps Engineer?",
#     "How many active users does the platform serve?",
#     "Which AWS services are used by the platform?",
#     "What database technologies are mentioned?",
#     "What frontend technology is used?",
#     "What backend technologies are used?",
#     "Which caching technology is implemented?",
#     "What deployment platform is used for microservices?",
#     "What percentage reduction in infrastructure costs was achieved?",
#     "What is the average API response time?",
#     "What is the average search response time?",
#     "How many API requests are processed each month?",
#     "How much did uptime improve after migration?",
#     "How long did the production incident last?",
#     "What caused the production incident?",
#     "When did the Redis incident occur?",
#     "What percentage of requests were affected during the incident?",
#     "How was the Redis issue resolved?",
#     "What was the root cause of the outage?",
#     "What security controls are implemented?",
#     "Does the platform use multi-factor authentication?",
#     "How is data encrypted?",
#     "What authorization model is used?",
#     "What authentication mechanism is used?"
# ]


In [31]:
system_prompt = """You are an advanced, intelligent assistant specialized in answering questions accurately based on a combination of retrieved context and your own extensive knowledge base.

When answering the user's query:
1. Prioritize any relevant, factual information provided in the "Retrieved Context" block below. 
2. If the retrieved context is brief, missing details, or highly specific, seamlessly blend in your own general knowledge to provide a comprehensive, clear, and well-rounded answer.
3. If the retrieved context is completely irrelevant to the user's true intent, rely entirely on your own accurate knowledge base to answer the question.
4. Maintain an objective, informative, and professional tone. Do not mention phrases like "According to the provided text" or "Based on the context" unless strictly necessary—make the response feel natural.

Retrieved Context:
{context}
"""

In [37]:
# for query in queries:
#     print(f"\n\nQUERY: {query}")
user_query = "total users"
results = retrieve_dynamic_section(user_query)

for idx, chunk in enumerate(results):
        print(f"\nResult {idx}")

        if isinstance(chunk, dict):
            print(chunk.get("text", ""))
        else:
            print(chunk)


Result 0
The organization currently serves more than 25,000 active users and processes approximately 1.2 million API requests per month. Search requests average 320 milliseconds, while standard API requests average 180 milliseconds.


In [28]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

llm = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")

In [38]:
from langchain_core.messages import SystemMessage, HumanMessage

context_str = ""
for idx, chunk in enumerate(results):
    chunk_text = chunk.get("text", "") if isinstance(chunk, dict) else str(chunk)
    context_str += f"[Result {idx+1}]: {chunk_text}\n\n"



# 3. Format the system prompt with the extracted context
formatted_system_prompt = system_prompt.format(context=context_str)

# 4. Compile messages for the Chat Model
messages = [
    SystemMessage(content=formatted_system_prompt),
    HumanMessage(content=user_query)
]

# 5. Invoke the Groq Model
try:
    print("Generating answer via Groq...")
    response = llm.invoke(messages)
    print("\n--- LLM ANSWER ---")
    print(response.content)
except Exception as e:
    print(f"Error generating answer: {e}")

Generating answer via Groq...

--- LLM ANSWER ---
The organization has more than 25,000 active users.
